In [1]:
pip install selenium beautifulsoup4 pandas webdriver-manager


Note: you may need to restart the kernel to use updated packages.Collecting selenium
     ---------------------------------------- 9.6/9.6 MB 36.0 MB/s eta 0:00:00
     ------------------------------------- 512.7/512.7 KB 31.4 MB/s eta 0:00:00
     ---------------------------------------- 82.6/82.6 KB 4.5 MB/s eta 0:00:00
     ------------------------------------- 182.8/182.8 KB 10.8 MB/s eta 0:00:00
     ---------------------------------------- 118.1/118.1 KB ? eta 0:00:00



You should consider upgrading via the 'c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [4]:
import sys
sys.stdout.write("HELLO\n")
sys.stdout.flush()


HELLO


In [2]:
1+1

2

In [1]:
import time
import itertools
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC



In [7]:
def start_driver():
    options = webdriver.ChromeOptions()

    # IMPORTANT : PAS de headless
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver


driver = start_driver()


In [8]:
driver.get("https://dpm.lol/tierlist?tier=gold_plus")


In [9]:
print("⏳ Attente du body...")

body = WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

print("✅ Body trouvé")
print(body.text[:500])



⏳ Attente du body...
✅ Body trouvé
Recherchez un Joueur, un Champion, une Équipe, un Pro...
Classé
Arena
ARAM
Tierlist & Builds Or+, 16.2
CHAMPIONS ANALYSÉS
66 184 920
?
Découvrez la tier list de League of Legends de DPM.LOL, mise à jour en direct avec les meilleurs champions pour chaque rôle. Accédez à des builds pro, des runes de haut niveau, des guides d’objets et des ordres de compétences basés sur des données pour améliorer votre rang et dominer vos parties !
GOLD+
TOUT
16.2
Rang
Champion
Voie
Tier
?
Winrate
Pickrate
Parties


In [9]:
buttons = driver.find_elements(By.TAG_NAME, "button")

for b in buttons:
    try:
        print(b.text)
    except:
        pass







GOLD+
TOUT
16.2
Rang
Champion
Voie
Tier
?
Winrate
Pickrate
Parties












Roadmap
Advertise with us
Contact
Privacy
ToS
Legal








In [13]:
container = driver.find_element(
    By.CSS_SELECTOR,
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

# récupérer les enfants directs
rows = container.find_elements(By.XPATH, "./div")
print(f"📊 Nombre de lignes détectées : {len(rows)}")

for i, row in enumerate(rows):
    print(f"\n🧱 Ligne {i}")
    print(row.text)


📊 Nombre de lignes détectées : 1

🧱 Ligne 0
1
Volibear
65.0%
S+
53.4%
8.9%
2
Braum
99.8%
S+
52.7%
10.4%
3
Diana
74.2%
S+
51.7%
11.5%
4
Jinx
99.7%
S+
52.0%
16.8%
5
Nami
99.9%
S+
52.1%
13.2%
6
Varus
36.9%
S+
52.6%
5.3%
7
Ekko
64.5%
S+
51.5%
9.0%
8
Miss Fortune
97.5%
S+
51.5%
12.6%
9
Malzahar
92.7%
S+
51.6%
7.9%
10
Swain
17.8%
S
54.4%
1.5%
11
Nilah
98.9%
S
54.1%
2.6%
12
Smolder
86.5%
S
51.2%
10.5%
13
Kayle
86.7%
S
52.8%
5.4%
14
Caitlyn
98.4%
S
50.0%
21.1%
15
Yasuo
72.9%
S
50.4%
10.8%
16
Milio
99.8%
S
51.8%
7.9%
17
Veigar
13.3%
S
54.4%
1.1%
18
Sona
99.6%
S
52.7%
6.8%
19
Rammus
94.5%
S
53.5%
2.5%
20
Twitch
90.6%
S
52.0%
6.5%


In [21]:
from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0  # temps pour laisser React rendre les lignes
scroll_step = 500    # pixels à scroller à chaque itération
all_rows_text = set()

# récupération du conteneur principal qui contient les lignes
container = driver.find_element(
    By.CSS_SELECTOR,
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md"
)

# Scroll de la page entière progressivement
last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    # récupérer toutes les lignes actuellement visibles
    rows = container.find_elements(By.XPATH, ".//div[contains(@class,'flex') and contains(@class,'gap-x-4')]")
    for row in rows:
        all_rows_text.add(row.text)

    # scroller la page entière
    scroll_top += scroll_step
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    # nouvelle hauteur
    new_height = driver.execute_script("return document.body.scrollHeight")

    # si on atteint le bas de la page, on sort
    if scroll_top >= new_height:
        break
    last_height = new_height

print(f"📊 Total de lignes récupérées : {len(all_rows_text)}\n")

# afficher quelques exemples
for i, line in enumerate(list(all_rows_text)[:20]):
    print(f"🧱 Ligne {i} :\n{line}\n")


📊 Total de lignes récupérées : 1

🧱 Ligne 0 :




In [24]:
from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0
scroll_step = 500
all_rows_text = set()

# Conteneur interne principal
container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    # scroller la page entière
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    # récupérer le conteneur interne
    container = driver.find_element(By.CSS_SELECTOR, container_selector)

    # récupérer toutes les divs enfants du type div:nth-child(x)
    child_divs = container.find_elements(By.XPATH, "./div/div")
    print(f"📊 Nombre de divs enfants trouvées à ce scroll : {len(child_divs)}")

    for div in child_divs:
        all_rows_text.add(div.text)

    # scroll vers le bas
    scroll_top += scroll_step
    new_height = driver.execute_script("return document.body.scrollHeight")

    # si on a atteint le bas, on s'arrête
    if scroll_top >= new_height:
        break
    last_height = new_height

print(f"\n📊 Total de lignes récupérées : {len(all_rows_text)}\n")

# afficher quelques exemples
for i, line in enumerate(list(all_rows_text)):
    print(f"🧱 Ligne {i} :\n{line}\n")


📊 Nombre de divs enfants trouvées à ce scroll : 20
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 34
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 28
📊 Nombre de divs enfants trouvées à ce scroll : 28

📊 Total de lignes récupérées : 212

🧱 Ligne 0 :
46
Gwen
54.1%
A
50.8%
4.0%

🧱 Lig

In [27]:
import pandas as pd
import re

champions_data = []

for line in all_rows_text:
    parts = line.split("\n")
    i = 0
    while i < len(parts):
        champ_name = parts[i].strip()
        # on vérifie qu'il y a bien des chiffres après le nom
        if i+4 < len(parts):
            try:
                winrate = float(parts[i+1].strip().replace('%',''))
                grade = parts[i+2].strip()
                pickrate = float(parts[i+3].strip().replace('%',''))
                banrate = float(parts[i+4].strip().replace('%',''))

                champions_data.append({
                    "Champion": champ_name,
                    "Winrate": winrate,
                    "Grade": grade,
                    "Pickrate": pickrate,
                    "Banrate": banrate
                })
                i += 5  # passer au prochain champion
            except ValueError:
                # si ça échoue, probablement un nom au milieu d'une ligne
                i += 1
        else:
            # on est à la fin de la ligne
            break

# créer le DataFrame
df = pd.DataFrame(champions_data)

# afficher le résultat
print(df.head(212))
print(f"\n📊 Total champions récupérés : {df.shape[0]}")


        Champion  Winrate Grade  Pickrate  Banrate
0           Gwen     54.1     A      50.8      4.0
1          Sylas     33.5     D      48.2      3.5
2           Ekko     33.9     A      50.9      4.7
3        Nidalee     76.3     D      47.5      1.2
4    Mordekaiser     88.2     A      51.0      6.6
..           ...      ...   ...       ...      ...
207       Kai'Sa     97.2     C      48.3     14.1
208         Kled     92.9     C      51.2      1.6
209     Pantheon     25.6     D      50.4      1.6
210   Cassiopeia     65.2     D      48.3      1.3
211        Urgot     96.7     C      51.2      3.1

[212 rows x 5 columns]

📊 Total champions récupérés : 212


In [28]:
import pandas as pd

# afficher toutes les lignes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)  # afficher toutes les colonnes
pd.set_option('display.width', 200)         # largeur totale
pd.set_option('display.max_colwidth', None) # afficher tout le contenu des colonnes

print(df)


         Champion  Winrate Grade  Pickrate  Banrate
0            Gwen     54.1     A      50.8      4.0
1           Sylas     33.5     D      48.2      3.5
2            Ekko     33.9     A      50.9      4.7
3         Nidalee     76.3     D      47.5      1.2
4     Mordekaiser     88.2     A      51.0      6.6
5          Maokai     75.4     D      50.1      1.4
6          Irelia     57.5     B      50.3      4.0
7          Gragas     60.7     D      48.2      1.5
8           Jayce     31.8     D      48.0      2.1
9          Aatrox     80.0     C      49.5      5.5
10           Sion     78.9     C      50.8      4.3
11        Vel'Koz     30.0     C      51.6      1.4
12         Lillia     96.6     D      49.9      2.8
13          Neeko     78.5     D      48.7      2.6
14        Smolder     86.5     S      51.2     10.5
15          Varus     12.4     A      51.8      1.8
16            Zed     35.4     C      49.6      4.2
17         Wukong     73.9     D      50.4      2.4
18          

In [ ]:
#petit probleme : ce n'est pas l'icone du role que l'on récupère ici mais celle du champion

from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0
scroll_step = 500

# on stocke maintenant des tuples (text, role_icon)
all_rows_data = set()

container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    container = driver.find_element(By.CSS_SELECTOR, container_selector)
    child_divs = container.find_elements(By.XPATH, "./div/div")

    print(f"📊 Nombre de divs enfants trouvées à ce scroll : {len(child_divs)}")

    for div in child_divs:
        try:
            text = div.text.strip()

            # 🔍 récupération de l'icône de rôle
            role_icon = None
            try:
                img = div.find_element(By.XPATH, ".//img")
                role_icon = (
                    img.get_attribute("alt")
                    or img.get_attribute("title")
                    or img.get_attribute("src")
                )
            except:
                role_icon = None

            if text:
                all_rows_data.add((text, role_icon))

        except:
            continue

    scroll_top += scroll_step
    new_height = driver.execute_script("return document.body.scrollHeight")

    if scroll_top >= new_height:
        break

print(f"\n📊 Total de lignes récupérées : {len(all_rows_data)}\n")


📊 Nombre de divs enfants trouvées à ce scroll : 20
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 34
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 28
📊 Nombre de divs enfants trouvées à ce scroll : 28

📊 Total de lignes récupérées : 212



In [36]:
for i, (text, role_icon) in enumerate(all_rows_data):
    print(f"🧱 Ligne {i}")
    print(text)
    print(f"🎯 Rôle (icône) : {role_icon}")
    print("-" * 40)


🧱 Ligne 0
44
Soraka
98.5%
A
51.7%
5.4%
🎯 Rôle (icône) : Soraka
----------------------------------------
🧱 Ligne 1
7
Ekko
64.5%
S+
51.5%
9.0%
🎯 Rôle (icône) : Ekko
----------------------------------------
🧱 Ligne 2
142
Renata
99.1%
D
50.6%
1.2%
🎯 Rôle (icône) : Renata
----------------------------------------
🧱 Ligne 3
186
Jayce
31.8%
D
48.0%
2.1%
🎯 Rôle (icône) : Jayce
----------------------------------------
🧱 Ligne 4
55
Singed
87.8%
B
52.6%
2.4%
🎯 Rôle (icône) : Singed
----------------------------------------
🧱 Ligne 5
53
Malphite
8.4%
A
51.4%
1.1%
🎯 Rôle (icône) : Malphite
----------------------------------------
🧱 Ligne 6
92
Brand
61.5%
C
50.7%
3.8%
🎯 Rôle (icône) : Brand
----------------------------------------
🧱 Ligne 7
164
Draven
95.9%
D
47.4%
3.7%
🎯 Rôle (icône) : Draven
----------------------------------------
🧱 Ligne 8
105
Shen
77.3%
C
50.9%
3.5%
🎯 Rôle (icône) : Shen
----------------------------------------
🧱 Ligne 9
35
Swain
21.2%
A
52.8%
1.8%
🎯 Rôle (icône) : Swain
--------

In [40]:
from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0
scroll_step = 500

# on stocke maintenant (texte_ligne, role_svg_info)
all_rows_data = set()

# Conteneur interne principal
container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    # scroller la page entière
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    # récupérer le conteneur interne
    container = driver.find_element(By.CSS_SELECTOR, container_selector)

    # récupérer toutes les lignes (champions)
    child_divs = container.find_elements(By.XPATH, "./div/div")
    print(f"📊 Nombre de divs enfants trouvées à ce scroll : {len(child_divs)}")

    for div in child_divs:
        line_text = div.text.strip()
        role_info = None

        # récupération de l’icône de rôle (svg dans le 9e bloc)
        try:
            role_svg = div.find_element(
                By.XPATH,
                ".//div[position()=9]//svg"
            )

            # on récupère ce qui est exploitable sur le svg
            role_info = (
                role_svg.get_attribute("aria-label")
                or role_svg.get_attribute("data-role")
                or role_svg.get_attribute("class")
            )

        except:
            role_info = None

        all_rows_data.add((line_text, role_info))

    # scroll vers le bas
    scroll_top += scroll_step
    new_height = driver.execute_script("return document.body.scrollHeight")

    # si on a atteint le bas, on s'arrête
    if scroll_top >= new_height:
        break

    last_height = new_height

print(f"\n📊 Total de lignes récupérées : {len(all_rows_data)}\n")

# afficher quelques exemples
for i, (text, role) in enumerate(list(all_rows_data)[:5]):
    print(f"🧱 Ligne {i} :")
    print(text)
    print(f"🎯 ROLE SVG RAW : {role}")
    print()


📊 Nombre de divs enfants trouvées à ce scroll : 20
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 34
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 33
📊 Nombre de divs enfants trouvées à ce scroll : 28
📊 Nombre de divs enfants trouvées à ce scroll : 28

📊 Total de lignes récupérées : 212

🧱 Ligne 0 :
90
Janna
99.0%
C
51.3%
3.6%
🎯 ROL

In [39]:
for text, role in list(all_rows_data)[:5]:
    print(text)
    print("🎯 role brut :", role)
    print("-" * 40)


90
Janna
99.0%
C
51.3%
3.6%
🎯 role brut : None
----------------------------------------
101
Kai'Sa
97.2%
C
48.3%
14.1%
🎯 role brut : None
----------------------------------------
8
Miss Fortune
97.5%
S+
51.5%
12.6%
🎯 role brut : None
----------------------------------------
58
Sivir
99.3%
B
50.7%
6.6%
🎯 role brut : None
----------------------------------------
89
Darius
94.6%
C
48.7%
7.0%
🎯 role brut : None
----------------------------------------


In [ ]:
#pas mal mais récupère en double
from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0
scroll_step = 500

# on stocke : (texte_ligne, [svg1, svg2, ...])
all_rows_data = []

container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    container = driver.find_element(By.CSS_SELECTOR, container_selector)

    # CHAQUE div ici = 1 champion
    rows = container.find_elements(By.XPATH, "./div/div")
    print(f"📊 Lignes détectées : {len(rows)}")

    for row in rows:
        row_text = row.text.strip()

        # 🔥 récupérer TOUS les svg sans exception
        svgs = row.find_elements(By.TAG_NAME, "svg")
        svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

        all_rows_data.append({
            "text": row_text,
            "svgs": svg_html_list
        })

    scroll_top += scroll_step
    new_height = driver.execute_script("return document.body.scrollHeight")

    if scroll_top >= new_height:
        break

print(f"\n📊 Total lignes collectées : {len(all_rows_data)}\n")


📊 Lignes détectées : 19
📊 Lignes détectées : 32
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 34
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 29
📊 Lignes détectées : 29

📊 Total lignes collectées : 572



In [11]:
for i, row in enumerate(all_rows_data):
    print(f"\n🧱 LIGNE {i}")
    print(row["text"])
    print(f"🖼️ Nombre de SVG trouvés : {len(row['svgs'])}")

    for j, svg in enumerate(row["svgs"]):
        print(f"\n--- SVG {j} ---")
        print(svg)



🧱 LIGNE 0
1
Volibear
65.0%
S+
53.4%
+1.9%
8.9%
587 928
🖼️ Nombre de SVG trouvés : 2

--- SVG 0 ---
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" fill="currentColor" class="w-[20px] h-[20px] text-black-300"><g fill-rule="evenodd"><g fill-rule="nonzero"><g><path d="M5.14 2c1.58 1.21 5.58 5.023 6.976 9.953s0 10.047 0 10.047c-2.749-3.164-5.893-5.2-6.18-5.382l-.02-.013C5.45 13.814 3 8.79 3 8.79c3.536.867 4.93 4.279 4.93 4.279C7.558 8.698 5.14 2 5.14 2zm14.976 5.907s-1.243 2.471-1.814 4.604c-.235.878-.285 2.2-.29 3.058v.282c.003.347.01.568.01.568s-1.738 2.397-3.38 3.678c.088-1.601.062-3.435-.208-5.334.928-2.023 2.846-5.454 5.682-6.856zm-2.124-5.331s-2.325 3.052-2.836 6.029c-.11.636-.201 1.194-.284 1.695-.379.584-.73 1.166-1.05 1.733-.033-.125-.06-.25-.095-.375-.302-1.07-.704-2.095-1.16-3.08.053-.146.103-.29.17-.438 0 0 1.814-3.78 5.255-5.564z" transform="translate(-2164.000000, -763.000000) translate(2164.000000, 763.000000)"></path></g></g></g></svg>

--- SVG 1 ---
<svg xmlns

In [44]:
import hashlib
from collections import defaultdict

role_svg_map = defaultdict(list)

for row in all_rows_data:
    if len(row["svgs"]) == 0:
        continue

    role_svg = row["svgs"][0]  # SVG DU ROLE
    svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()

    role_svg_map[svg_hash].append(row["text"])


In [45]:
print(f"Nombre de rôles détectés : {len(role_svg_map)}")

for h, rows in role_svg_map.items():
    print("\nROLE HASH:", h)
    print("Exemple de champion:")
    print(rows[0])


Nombre de rôles détectés : 5

ROLE HASH: 54d7bacd7686d25f9555c3381d5b3ecb
Exemple de champion:
1
Volibear
65.0%
S+
53.4%
8.9%

ROLE HASH: d1a365179625b6191d515c69f5277dbd
Exemple de champion:
2
Braum
99.8%
S+
52.7%
10.4%

ROLE HASH: f84094b0fe98e4bf44fe62648e255e41
Exemple de champion:
4
Jinx
99.7%
S+
52.0%
16.8%

ROLE HASH: 6f7f06ca1bef87e71a35726cf843ad87
Exemple de champion:
6
Varus
36.9%
S+
52.6%
5.3%

ROLE HASH: e4f796e42865301ea9dd362f979a2cdc
Exemple de champion:
9
Malzahar
92.7%
S+
51.6%
7.9%


---
on y est presque
---

In [57]:
from selenium.webdriver.common.by import By
import re

meta_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)

meta_container = driver.find_element(By.CSS_SELECTOR, meta_container_selector)

# 🔍 on récupère TOUS les spans visibles
spans = meta_container.find_elements(By.TAG_NAME, "span")
span_texts = [s.text.strip() for s in spans if s.text.strip()]

print("🧪 Spans détectés :", span_texts)

# 🎯 identification intelligente
elo = None
server = None
patch = None

for text in span_texts:
    if re.match(r"^\d+\.\d+$", text):          # ex: 14.2
        patch = text
    elif text.upper() in {"EUW", "KR", "NA", "BR", "EUNE", "JP", "LAN", "LAS", "OCE", "RU", "TR", "VN", "TOUT"}:
        server = text
    else:
        elo = text

print("📌 META DETECTÉE")
print(f"   🎯 Elo    : {elo}")
print(f"   🌍 Server : {server}")
print(f"   🧩 Patch  : {patch}")


🧪 Spans détectés : ['GOLD+', 'GOLD+', 'TOUT', 'TOUT', '16.2']
📌 META DETECTÉE
   🎯 Elo    : GOLD+
   🌍 Server : TOUT
   🧩 Patch  : 16.2


In [ ]:
# Ce code commenté marche très bien mais il manque le elo, le patch, et le serveur.

from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0
scroll_step = 500

# stockage final
all_rows_data = []

# 🔒 set pour éviter les doublons
seen_champions = set()

container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    container = driver.find_element(By.CSS_SELECTOR, container_selector)

    # chaque div = 1 champion
    rows = container.find_elements(By.XPATH, "./div/div")
    print(f"📊 Lignes détectées : {len(rows)}")

    for row in rows:
        row_text = row.text.strip()

        # ⛔ déjà vu → on skip
        if row_text in seen_champions:
            continue

        seen_champions.add(row_text)

        svgs = row.find_elements(By.TAG_NAME, "svg")
        svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

        all_rows_data.append({
            "text": row_text,
            "svgs": svg_html_list
        })

    scroll_top += scroll_step
    new_height = driver.execute_script("return document.body.scrollHeight")

    if scroll_top >= new_height:
        break

print(f"\n✅ Total champions uniques collectés : {len(all_rows_data)}\n")


📊 Lignes détectées : 19
📊 Lignes détectées : 32
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 34
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 29
📊 Lignes détectées : 29

✅ Total champions uniques collectés : 212



In [20]:
for i, row in enumerate(all_rows_data):
    print(f"\n🧱 LIGNE {i}")
    print(row["text"])
    print(f"🖼️ Nombre de SVG trouvés : {len(row['svgs'])}")

    for j, svg in enumerate(row["svgs"]):
        print(f"\n--- SVG {j} ---")
        print(svg)


🧱 LIGNE 0
1
Volibear
65.0%
S+
53.4%
+1.9%
8.9%
589 850
🖼️ Nombre de SVG trouvés : 2

--- SVG 0 ---
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" fill="currentColor" class="w-[20px] h-[20px] text-black-300"><g fill-rule="evenodd"><g fill-rule="nonzero"><g><path d="M5.14 2c1.58 1.21 5.58 5.023 6.976 9.953s0 10.047 0 10.047c-2.749-3.164-5.893-5.2-6.18-5.382l-.02-.013C5.45 13.814 3 8.79 3 8.79c3.536.867 4.93 4.279 4.93 4.279C7.558 8.698 5.14 2 5.14 2zm14.976 5.907s-1.243 2.471-1.814 4.604c-.235.878-.285 2.2-.29 3.058v.282c.003.347.01.568.01.568s-1.738 2.397-3.38 3.678c.088-1.601.062-3.435-.208-5.334.928-2.023 2.846-5.454 5.682-6.856zm-2.124-5.331s-2.325 3.052-2.836 6.029c-.11.636-.201 1.194-.284 1.695-.379.584-.73 1.166-1.05 1.733-.033-.125-.06-.25-.095-.375-.302-1.07-.704-2.095-1.16-3.08.053-.146.103-.29.17-.438 0 0 1.814-3.78 5.255-5.564z" transform="translate(-2164.000000, -763.000000) translate(2164.000000, 763.000000)"></path></g></g></g></svg>

--- SVG 1 ---
<svg xmlns

In [26]:
ROLE_HASH_TO_TEXT = {
    "54d7bacd7686d25f9555c3381d5b3ecb": "jungle",
    "d1a365179625b6191d515c69f5277dbd": "support",
    "f84094b0fe98e4bf44fe62648e255e41": "adc",
    "6f7f06ca1bef87e71a35726cf843ad87": "top",
    "e4f796e42865301ea9dd362f979a2cdc": "mid",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role



In [28]:
import hashlib
from collections import defaultdict

role_svg_map = defaultdict(list)

print(f"🔍 Début traitement de {len(all_rows_data)} champions\n")

for idx, row in enumerate(all_rows_data, start=1):
    print(f"➡️ [{idx}] Champion : {row['text'][:60]}...")

    if len(row["svgs"]) == 0:
        print("   ⛔ Aucun SVG trouvé → skip\n")
        continue

    role_svg = row["svgs"][0]  # SVG DU ROLE
    svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
    print(svg_hash)
    print(resolve_role_from_hash(svg_hash))

    # nouveau rôle détecté
    if svg_hash not in role_svg_map:
        print(f"   🆕 Nouveau rôle détecté (hash={svg_hash})")

    role_svg_map[svg_hash].append(row["text"])
    print(f"   ✅ Ajouté au rôle {svg_hash[:8]} "
          f"(total: {len(role_svg_map[svg_hash])})\n")

print("\n📊 RÉSUMÉ FINAL")
print(f"👉 Nombre de rôles distincts : {len(role_svg_map)}")

for i, (svg_hash, champions) in enumerate(role_svg_map.items(), start=1):
    print(f"\n🎭 Rôle #{i} — {resolve_role_from_hash(svg_hash)}")
    print(f"   Champions ({len(champions)}) :")
    for champ in champions:
        print(f"    • {champ}")


🔍 Début traitement de 212 champions

➡️ [1] Champion : 1
Volibear
65.0%
S+
53.4%
+1.9%
8.9%
589 850...
54d7bacd7686d25f9555c3381d5b3ecb
jungle
   🆕 Nouveau rôle détecté (hash=54d7bacd7686d25f9555c3381d5b3ecb)
   ✅ Ajouté au rôle 54d7bacd (total: 1)

➡️ [2] Champion : 2
Braum
99.8%
S+
52.8%
+1.9%
10.4%
687 584...
d1a365179625b6191d515c69f5277dbd
support
   🆕 Nouveau rôle détecté (hash=d1a365179625b6191d515c69f5277dbd)
   ✅ Ajouté au rôle d1a36517 (total: 1)

➡️ [3] Champion : 3
Diana
74.2%
S+
51.7%
+0.5%
11.5%
761 662...
54d7bacd7686d25f9555c3381d5b3ecb
jungle
   ✅ Ajouté au rôle 54d7bacd (total: 2)

➡️ [4] Champion : 4
Jinx
99.7%
S+
52.0%
+0.9%
16.8%
1 115 761...
f84094b0fe98e4bf44fe62648e255e41
adc
   🆕 Nouveau rôle détecté (hash=f84094b0fe98e4bf44fe62648e255e41)
   ✅ Ajouté au rôle f84094b0 (total: 1)

➡️ [5] Champion : 5
Nami
99.9%
S+
52.1%
-0.3%
13.1%
871 985...
d1a365179625b6191d515c69f5277dbd
support
   ✅ Ajouté au rôle d1a36517 (total: 2)

➡️ [6] Champion : 6
Varus
37.0%
S+
52.6

In [29]:
print(f"Nombre de rôles détectés : {len(role_svg_map)}")

for h, rows in role_svg_map.items():
    print("\nROLE HASH:", h)
    print("Exemple de champion:")
    print(rows[0])

Nombre de rôles détectés : 5

ROLE HASH: 54d7bacd7686d25f9555c3381d5b3ecb
Exemple de champion:
1
Volibear
65.0%
S+
53.4%
+1.9%
8.9%
589 850

ROLE HASH: d1a365179625b6191d515c69f5277dbd
Exemple de champion:
2
Braum
99.8%
S+
52.8%
+1.9%
10.4%
687 584

ROLE HASH: f84094b0fe98e4bf44fe62648e255e41
Exemple de champion:
4
Jinx
99.7%
S+
52.0%
+0.9%
16.8%
1 115 761

ROLE HASH: 6f7f06ca1bef87e71a35726cf843ad87
Exemple de champion:
6
Varus
37.0%
S+
52.6%
+3.1%
5.3%
352 432

ROLE HASH: e4f796e42865301ea9dd362f979a2cdc
Exemple de champion:
9
Malzahar
92.7%
S+
51.7%
+0.2%
7.8%
521 253


In [ ]:
def parse_champion_text(text: str) -> dict:
    """
    Attend un texte multi-lignes :
    1: nom du champion
    2: % présence dans le rôle
    3: tier (S+, S, A...)
    4: winrate
    5: pickrate
    6: nombre de parties
    """
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    if len(lines) < 6:
        print("⚠️ Format inattendu :", lines)
        return None

    return {
        "champion": lines[1],
        "role_pickrate": lines[2],
        "tier": lines[3],
        "winrate": lines[4],
        "pickrate": lines[6],
        "games": lines[7],
        "winrate+": lines[5],
    }


In [ ]:
import pandas as pd
import hashlib

rows_for_df = []

for row in all_rows_data:
    if len(row["svgs"]) == 0:
        continue

    parsed = parse_champion_text(row["text"])
    if parsed is None:
        continue

    role_svg = row["svgs"][0]
    svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
    role = resolve_role_from_hash(svg_hash)

    rows_for_df.append({
        "elo": "gold_plus",
        "server": "all",
        "patch": "latest",
        "champion": parsed["champion"],
        "role": role,
        "role_pickrate": parsed["role_pickrate"],
        "tier": parsed["tier"],
        "winrate": parsed["winrate"],
        "winrate_evol": parsed["winrate+"],
        "pickrate": parsed["pickrate"],
        "games": parsed["games"],
    })

df = pd.DataFrame(rows_for_df)


In [51]:
df.head()

,champion,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,Volibear,jungle,65.0%,S+,53.4%,+1.9%,8.9%,589 850
1,Braum,support,99.8%,S+,52.8%,+1.9%,10.4%,687 584
2,Diana,jungle,74.2%,S+,51.7%,+0.5%,11.5%,761 662
3,Jinx,adc,99.7%,S+,52.0%,+0.9%,16.8%,1 115 761
4,Nami,support,99.9%,S+,52.1%,-0.3%,13.1%,871 985
